# Item-Based Collaborative Filtering

## Objective

In this notebook, we implement an **Item-Based Collaborative Filtering (IBCF)** recommendation system.

Unlike User-Based Collaborative Filtering, which finds users with similar preferences, Item-Based Collaborative Filtering identifies movies that are similar based on user rating patterns.

The workflow includes:

- Loading the processed dataset
- Creating the User–Movie Interaction Matrix
- Computing Item Similarity using Cosine Similarity
- Finding similar movies
- Generating personalized movie recommendations

Item-Based Collaborative Filtering is widely used because item similarities remain relatively stable over time, making it more scalable for large recommendation systems.

## What is Item-Based Collaborative Filtering?

Item-Based Collaborative Filtering recommends movies based on the similarity between items rather than the similarity between users.

### Example

Suppose many users who liked:

- Toy Story
- Finding Nemo
- Monsters Inc.

also liked

- Cars

Then if a user watches **Toy Story**, the recommendation system may recommend **Cars**, even if the user has never watched it before.

Instead of asking:

> "Which users are similar?"

Item-Based Collaborative Filtering asks:

> "Which movies are similar?"

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
movie_ratings = pd.read_csv(
    "processed_data/movie_ratings_processed.csv"
)

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
0,196,242,3,881250949,Kolya (1996),195,241
1,186,302,3,891717742,L.A. Confidential (1997),185,301
2,22,377,1,878887116,Heavyweights (1994),21,376
3,244,51,2,880606923,Legends of the Fall (1994),243,50
4,166,346,1,886397596,Jackie Brown (1997),165,345


## Step 1: Create the User–Movie Interaction Matrix

Each row represents a user.

Each column represents a movie.

Each value represents the rating given by a user.

Missing ratings are replaced with **0** because cosine similarity cannot operate on missing (`NaN`) values.

The interaction matrix forms the foundation for computing movie-to-movie similarity.

In [3]:
user_movie_matrix = movie_ratings.pivot_table(
    index="user_id",
    columns="movie_title",
    values="rating"
)

user_movie_matrix = user_movie_matrix.fillna(0)

user_movie_matrix.head()

movie_title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,2.0,5.0,0.0,0.0,3.0,4.0,0.0,0.0,...,0.0,0.0,0.0,5.0,3.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0


## Why Do We Transpose the Matrix?

For User-Based Collaborative Filtering:

- Rows → Users
- Columns → Movies

To compare users, this representation is sufficient.

However, for Item-Based Collaborative Filtering, we want to compare **movies**, not users.

Therefore, we transpose the matrix.

Before Transpose:

Rows → Users

Columns → Movies

After Transpose:

Rows → Movies

Columns → Users

Now each movie is represented by a vector of user ratings, allowing us to compute similarity between movies.

In [4]:
movie_user_matrix = user_movie_matrix.T

movie_user_matrix.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
movie_title,,,,,,,,,,,,,,,,,,,,,
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1-900 (1994),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
101 Dalmatians (1996),2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,2.0,0.0,0.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0
12 Angry Men (1957),5.0,0.0,0.0,0.0,0.0,4.0,4.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
187 (1997),0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 2: Compute Item Similarity

Each movie is represented by its user rating vector.

Cosine Similarity compares these vectors to identify movies with similar rating patterns.

Movies that receive similar ratings from many users are considered similar and can be recommended together.

In [5]:
item_similarity = cosine_similarity(movie_user_matrix)

print("Item Similarity Matrix Shape:")
print(item_similarity.shape)

Item Similarity Matrix Shape:
(1664, 1664)


In [6]:
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

item_similarity_df.head()

movie_title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
movie_title,,,,,,,,,,,,,,,,,,,,,
'Til There Was You (1997),1.000000,0.000000,0.024561,0.099561,0.185236,0.159265,0.000000,0.052203,0.000000,0.033326,...,0.000000,0.000000,0.000000,0.027774,0.118840,0.142315,0.029070,0.000000,0.110208,0.000000
1-900 (1994),0.000000,1.000000,0.014139,0.009294,0.007354,0.004702,0.010055,0.067038,0.000000,0.000000,...,0.152499,0.015484,0.000000,0.069284,0.018243,0.023408,0.006694,0.079640,0.042295,0.000000
101 Dalmatians (1996),0.024561,0.014139,1.000000,0.167006,0.061105,0.143878,0.203781,0.225803,0.027642,0.092337,...,0.000000,0.021965,0.030905,0.274877,0.204267,0.101199,0.056976,0.172155,0.045714,0.000000
12 Angry Men (1957),0.099561,0.009294,0.167006,1.000000,0.056822,0.167235,0.304078,0.422506,0.072682,0.394854,...,0.060946,0.016502,0.000000,0.403270,0.259436,0.145519,0.105226,0.038901,0.060101,0.081261
187 (1997),0.185236,0.007354,0.061105,0.056822,1.000000,0.132327,0.042928,0.065060,0.043133,0.027300,...,0.000000,0.141997,0.000000,0.068257,0.067786,0.091293,0.099490,0.025184,0.142667,0.096449


## Understanding the Item Similarity Matrix

Each row and column represents a movie.

The similarity value indicates how closely two movies are related based on user rating behavior.

For example:

| Movie A | Movie B | Similarity |
|----------|----------|-----------:|
| Toy Story | Toy Story | 1.000 |
| Toy Story | Finding Nemo | 0.81 |
| Toy Story | Titanic | 0.29 |

### Interpretation

- **1.0** → Same movie (perfect similarity)
- Higher values indicate similar audience preferences.
- Lower values indicate different user preferences.

The Item Similarity Matrix is used to recommend movies that are similar to those a user has already enjoyed.

### Interview Question

**Q. What is the difference between User-Based and Item-Based Collaborative Filtering?**

**Answer:**

User-Based Collaborative Filtering finds users with similar preferences and recommends movies liked by those users.

Item-Based Collaborative Filtering finds movies with similar rating patterns and recommends movies similar to those the user has already watched.

Item-Based Collaborative Filtering is generally more scalable because item relationships change less frequently than user preferences.

# Understanding the Item Similarity Matrix

The Item Similarity Matrix stores the similarity score between every pair of movies.

Each row represents one movie.

Each column also represents one movie.

Every value indicates how similar two movies are based on users' rating patterns.

For example,

| Movie A | Movie B | Similarity |
|----------|----------|-----------:|
| Toy Story | Toy Story | 1.000 |
| Toy Story | Finding Nemo | 0.91 |
| Toy Story | Titanic | 0.28 |

## Why is the diagonal always 1?

Every movie is perfectly similar to itself.

Therefore,

```
Similarity(Movie A, Movie A) = 1
```

which explains why all diagonal entries of the similarity matrix are equal to **1.0**.

## Interpretation

- Similarity close to **1** → Movies are highly similar.
- Similarity around **0** → Movies are unrelated.
- The Item Similarity Matrix is later used to recommend movies similar to those already liked by a user.

# Finding Similar Movies

Now that we have computed the Item Similarity Matrix, we can retrieve movies that are most similar to a given movie.

The recommendation process is simple:

1. Select a movie.
2. Retrieve its similarity scores with all other movies.
3. Sort the similarity scores in descending order.
4. Remove the movie itself.
5. Return the Top-K most similar movies.

This forms the foundation of Item-Based Collaborative Filtering.

In [7]:
def get_similar_movies(movie_name, top_n=10):

    similar_movies = item_similarity_df[movie_name].sort_values(
        ascending=False
    )

    # Remove the movie itself
    similar_movies = similar_movies.drop(movie_name)

    return similar_movies.head(top_n)

In [8]:
get_similar_movies("Toy Story (1995)")

movie_title
Star Wars (1977)                                0.734572
Return of the Jedi (1983)                       0.699925
Independence Day (ID4) (1996)                   0.689786
Rock, The (1996)                                0.664555
Mission: Impossible (1996)                      0.641322
Willy Wonka and the Chocolate Factory (1971)    0.638158
Star Trek: First Contact (1996)                 0.636727
Fargo (1996)                                    0.630601
Jerry Maguire (1996)                            0.624075
Raiders of the Lost Ark (1981)                  0.622382
Name: Toy Story (1995), dtype: float64

In [9]:
get_similar_movies("Toy Story (1995)")

movie_title
Star Wars (1977)                                0.734572
Return of the Jedi (1983)                       0.699925
Independence Day (ID4) (1996)                   0.689786
Rock, The (1996)                                0.664555
Mission: Impossible (1996)                      0.641322
Willy Wonka and the Chocolate Factory (1971)    0.638158
Star Trek: First Contact (1996)                 0.636727
Fargo (1996)                                    0.630601
Jerry Maguire (1996)                            0.624075
Raiders of the Lost Ark (1981)                  0.622382
Name: Toy Story (1995), dtype: float64

# Understanding Similar Movie Recommendations

We queried the recommendation system with the movie:

```
Toy Story (1995)
```

The system returned the Top 10 most similar movies based on cosine similarity.

Each similarity score ranges from **0 to 1**.

- A score close to **1** indicates that two movies have very similar rating patterns.
- A score close to **0** indicates that the movies are not strongly related.

Example:

| Movie | Similarity |
|--------|-----------:|
| Star Wars (1977) | 0.7346 |
| Return of the Jedi (1983) | 0.6999 |
| Fargo (1996) | 0.6306 |

This means users who highly rated **Toy Story (1995)** also tended to highly rate these movies.

The recommendation is **not based on movie genres or descriptions**.

Instead, it is based entirely on **user behavior** (collaborative filtering).

## Interview Question

### Q. Does Item-Based Collaborative Filtering use movie genres?

**Answer:**

No.

Item-Based Collaborative Filtering does **not** use genres, actors, directors, or movie descriptions.

It recommends movies by analyzing historical user ratings.

If many users rate two movies similarly, the algorithm considers them similar, regardless of their genres.

# Personalized Item-Based Recommendation

The previous function recommended movies similar to a **single movie**.

However, users usually watch many movies.

In this section, we generate recommendations by considering **all movies watched by a user**.

### Recommendation Process

1. Identify all movies watched by the target user.
2. Find movies similar to each watched movie.
3. Combine similarity scores from all watched movies.
4. Remove movies already watched.
5. Rank the remaining movies by their total similarity score.
6. Recommend the Top-K unseen movies.

This approach produces personalized recommendations based on the user's complete viewing history.

In [10]:
def recommend_movies_item_based(user_id, top_n=10):

    # Movies watched by the user
    watched_movies = user_movie_matrix.loc[user_id]

    watched_movies = watched_movies[watched_movies > 0].index.tolist()

    recommendation_scores = {}

    for movie in watched_movies:

        similar_movies = item_similarity_df[movie]

        for similar_movie, score in similar_movies.items():

            if similar_movie not in watched_movies:

                recommendation_scores[similar_movie] = (
                    recommendation_scores.get(similar_movie, 0) + score
                )

    recommendations = (
        pd.Series(recommendation_scores)
        .sort_values(ascending=False)
        .head(top_n)
    )

    return recommendations

In [11]:
recommend_movies_item_based(user_id=1)

E.T. the Extra-Terrestrial (1982)         105.062429
Speed (1994)                               99.508648
Batman (1989)                              98.866387
True Lies (1994)                           98.659959
Stand by Me (1986)                         97.365360
Mrs. Doubtfire (1993)                      94.778195
Clueless (1995)                            94.021538
Schindler's List (1993)                    93.868687
Clear and Present Danger (1994)            92.095126
One Flew Over the Cuckoo's Nest (1975)     92.011135
dtype: float64

# Observations

The recommendation system generates different movie recommendations for different users based on their watch history.

Unlike the previous function, which recommended movies similar to a single movie, this approach considers **all movies watched by a user**.

The similarity scores from each watched movie are aggregated to produce the final recommendation list.

This results in a more personalized recommendation system.

Different users receive different recommendations because their viewing histories are different.

# Conclusion

In this notebook, we successfully implemented an **Item-Based Collaborative Filtering Recommendation System**.

## Tasks Completed

- Loaded the processed MovieLens dataset.
- Created the User–Movie Interaction Matrix.
- Computed Item Similarity using Cosine Similarity.
- Retrieved similar movies for any selected movie.
- Built a personalized recommendation engine using item similarity.
- Recommended unseen movies based on the user's watch history.

## Advantages of Item-Based Collaborative Filtering

- More scalable than User-Based Collaborative Filtering.
- Item similarities remain relatively stable over time.
- Faster to serve recommendations in production.
- Widely adopted in industrial recommendation systems.

## Limitation

This approach relies only on historical user ratings and cannot recommend movies with no rating history (Cold Start Problem).

In the next notebook, we will overcome some of these limitations using **Matrix Factorization (SVD)**.

## Interview Question

### Q. Why is Item-Based Collaborative Filtering preferred over User-Based Collaborative Filtering?

**Answer:**

Item-Based Collaborative Filtering is generally preferred because item relationships change less frequently than user preferences.

Items (movies, products, books) are relatively stable, while users continuously join the platform and change their interests.

This makes Item-Based Collaborative Filtering more scalable and computationally efficient for large recommendation systems.

# Understanding Recommendation Scores

The values returned by the Item-Based Recommendation System are **recommendation scores**, not movie ratings.

For example:

| Movie | Recommendation Score |
|--------|---------------------:|
| E.T. the Extra-Terrestrial | 105.06 |
| Speed | 99.51 |

These scores are obtained by summing the similarity scores of movies that the user has already watched.

A higher score indicates that a movie is strongly related to many movies in the user's watch history.

Therefore,

- Higher Score → Stronger Recommendation
- Lower Score → Weaker Recommendation

These scores do **not** represent IMDb ratings or predicted user ratings.

## Interview Question

### Q. What does the recommendation score represent?

**Answer:**

The recommendation score represents the cumulative similarity between a candidate movie and the movies already watched by the user.

It is not a movie rating.

A higher recommendation score indicates that the movie is more strongly connected to the user's viewing history and is therefore more likely to be relevant.